In [1]:

from datasets import load_dataset

# Load the dataset
dataset = load_dataset("ButterChicken98/plantvillage-image-text-pairs")

# Let's see what's inside
print(dataset)

c:\Users\SREELAKSHMI\Documents\plant disease detection\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['image', 'caption', 'captions'],
        num_rows: 20638
    })
})


libraries


In [2]:
import torch
import pandas as pd
import numpy as np

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import evaluate

CONVERT TO CSV

In [3]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset("imdb")

# Convert to pandas
df = dataset["train"].to_pandas()

# Save as CSV
df.to_csv("dataset.csv", index=False)

LOAD CSV AND RENAME

In [4]:
df = pd.read_csv("dataset.csv")

print("Original Columns:", df.columns)

# Rename properly
df = df.rename(columns={
    "caption": "label",
    "captions": "text"
})

print("Renamed Columns:", df.columns)
print("Total Rows:", len(df))

df.head()

Original Columns: Index(['text', 'label'], dtype='str')
Renamed Columns: Index(['text', 'label'], dtype='str')
Total Rows: 25000


,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


ENCODE LABELS

In [5]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["label"])

num_classes = len(le.classes_)
class_names = list(le.classes_)

print("Number of classes:", num_classes)
print("Class names:", class_names)


Number of classes: 2
Class names: [np.int64(0), np.int64(1)]


TRAIN TEST SPLIT

In [6]:
df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Train Size:", len(df_train))
print("Test Size:", len(df_test))

Train Size: 20000
Test Size: 5000


CONVERT TO HUGGINGFACE DATASET

In [7]:
train_dataset = Dataset.from_pandas(df_train, preserve_index=False)
test_dataset = Dataset.from_pandas(df_test, preserve_index=False)

LOAD TOKENIZER

In [8]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

TOKENIZE DATASET

In [9]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding=False,
        max_length=128
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

Map: 100%|██████████| 5000/5000 [00:00<00:00, 5915.56 examples/s]


LOAD MODEL

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3621.84it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


DATA COLLATOR

In [11]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

METRICS


In [12]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

TRAINING ARGUMENTS

In [13]:
import sys
import importlib

# Reload transformers to use the newly installed version
if 'transformers' in sys.modules:
    importlib.reload(sys.modules['transformers'])

from transformers import TrainingArguments

In [14]:
import torch
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    fp16=torch.cuda.is_available()
)

In [15]:
import transformers
print(transformers.__version__)

5.3.0


TRAINER


In [16]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

TRAIN


In [17]:
trainer.train()

c:\Users\SREELAKSHMI\Documents\plant disease detection\myenv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.374013,0.318983,0.870400
2,0.219383,0.328317,0.878200
3,0.106427,0.498081,0.880400
4,0.047041,0.614727,0.880200
5,0.018724,0.704480,0.879200


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.83it/s]
c:\Users\SREELAKSHMI\Documents\plant disease detection\myenv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.19it/s]
c:\Users\SREELAKSHMI\Documents\plant disease detection\myenv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.47it/s]
c:\Users\SREELAKSHMI\Documents\plant disease detection\myenv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing mo

TrainOutput(global_step=6250, training_loss=0.15311767364501952, metrics={'train_runtime': 18494.1527, 'train_samples_per_second': 5.407, 'train_steps_per_second': 0.338, 'total_flos': 3311684966400000.0, 'train_loss': 0.15311767364501952, 'epoch': 5.0})

SAVE MODEL

In [18]:
save_dir = "./distilbert_plant_model"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("Model saved successfully ✅")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.11it/s]

Model saved successfully ✅


In [20]:
save_dir = "./distilbert_plant_model"

# Convert labels to string to avoid JSON issues
model.config.id2label = {int(i): str(name) for i, name in enumerate(class_names)}
model.config.label2id = {str(name): int(i) for i, name in enumerate(class_names)}

# Save model
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("Model saved successfully! ✅")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.98it/s]

Model saved successfully! ✅


INFERENCE TEST

In [21]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(save_dir)
tokenizer = AutoTokenizer.from_pretrained(save_dir)

model.to(device)
model.eval()

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        pred_id = outputs.logits.argmax(dim=-1).item()
    return class_names[pred_id]

print(predict("Leaves show brown circular spots with yellow halo."))

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 5924.80it/s]


1
